# [5.5] Diffusion Language Models - Solutions

Reference validation notebook for the section-local discrete diffusion implementation. This executes the visible sub-function tests against `solutions.py` and then runs the CPU notebook contract. The CUDA report is available through `solutions.run_gpu_test(max_vram_gb=24.0)`.

Expected CUDA highlights: trained tiny conditional denoiser passes, held-out masked accuracy at least 0.95, sampler exact match at least 0.95, shuffled-label accuracy at most 0.25, activation trajectory shapes valid, and peak VRAM below 1GB for this tiny path. The released-checkpoint proof is separate: Google BF16 direct local loading is deferred for the 24GB tier, while the pinned NVIDIA NVFP4 checkpoint has an isolated vLLM 0.24.0 generation artifact on the RTX 5090 Laptop GPU.


In [ ]:
import sys
from pathlib import Path

chapter = "chapter5_modern_architectures"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_diffusion_language_models.tests as tests
from part5_diffusion_language_models import solutions


In [ ]:
tests.test_linear_mask_schedule_and_expected_fraction(
    solutions.linear_mask_schedule,
    solutions.expected_mask_fraction,
)
tests.test_forward_noising_extremes_and_seeded_masks(
    solutions.apply_forward_noising,
    solutions.linear_mask_schedule,
)
tests.test_masked_denoising_loss_uses_only_masked_positions(
    solutions.masked_denoising_loss,
)
tests.test_confidence_remask_entropy_and_uniform_control(
    solutions.confidence_remask,
    solutions.token_entropy,
    solutions.uniform_remask,
)
tests.test_oracle_diffusion_sampler_recovers_target(
    solutions.diffusion_sampler,
    solutions.linear_mask_schedule,
)
tests.test_commitment_edit_distance_and_activation_trajectory(
    solutions.commitment_times,
    solutions.edit_distance,
    solutions.validate_activation_trajectory,
)
tests.test_tiny_conditional_diffusion_lm_forward_shape(
    solutions.TinyConditionalDiffusionLM,
)
tests.test_notebook_contract(solutions.run_smoke_test)
tests.test_diffusiongemma_vllm_probe_artifact_contract()


In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
smoke
